# 📘 Ejemplo para desarrollar en clase
## Pre-entrega: Adquisición y Normalización con Pandas — Tienda "ElectroSur"

Este notebook resuelve, paso a paso, la consigna de la pre-entrega:

> Simular un escenario real donde los datos de ventas provienen de distintos departamentos en formatos diferentes (un CSV de transacciones y un Excel con el catálogo de productos), integrando y normalizando la información.

**Escenario:** ElectroSur es una tienda de electrónica. El área de ventas registra las transacciones en un sistema que exporta a **CSV**, mientras que el área de compras mantiene el catálogo de productos en una planilla de **Excel**.

**Archivos de este ejemplo** (deben estar en la misma carpeta que este notebook):
- `transacciones_electrosur.csv` → `id_transaccion`, `id_producto`, `cantidad`, `fecha`
- `productos_electrosur.xlsx` → `id_producto`, `nombre_producto`, `precio_unitario`, `categoria`

**Vamos a seguir los mismos 5 pasos que pide la consigna:**

1. Adquisición (cargar CSV y Excel)
2. Transformación (tipos de datos, nulos, merge)
3. Normalización (columna `total_venta`)
4. Validación
5. Exportación a Parquet


## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

RUTA_TRANSACCIONES = Path("transacciones_electrosur.csv")
RUTA_PRODUCTOS = Path("productos_electrosur.xlsx")
RUTA_SALIDA = Path("ventas_electrosur_consolidado.parquet")

print("Archivos disponibles:", RUTA_TRANSACCIONES.exists(), RUTA_PRODUCTOS.exists())


## 1. Adquisición

Cargamos cada fuente con la función de pandas correspondiente. Nota: `pd.read_excel` necesita el motor `openpyxl` instalado (`pip install openpyxl`) — si no lo tenés instalado vas a ver un `ImportError`, uno de los "errores comunes" que menciona la consigna.

In [ ]:
transacciones = pd.read_csv(RUTA_TRANSACCIONES)
productos = pd.read_excel(RUTA_PRODUCTOS)  # requiere openpyxl

print("Transacciones:", transacciones.shape)
print("Productos:", productos.shape)
transacciones.head()


In [ ]:
productos.head()


## 2. Transformación

### 2.1 Exploración inicial — antes de tocar nada, observamos

Aplicamos el mismo criterio del EDA: `info()`, `isnull().sum()`, y revisamos los `dtypes` de la columna que vamos a usar como llave del merge (`id_producto`).

In [ ]:
transacciones.info()


In [ ]:
print("Nulos por columna (transacciones):")
print(transacciones.isnull().sum())
print()
print("dtype id_producto en transacciones:", transacciones["id_producto"].dtype)
print("dtype id_producto en productos:   ", productos["id_producto"].dtype)


**Buena noticia:** en este caso ambos `id_producto` ya son `int64`, así que el merge no debería romperse por tipos incompatibles. (En el notebook de práctica vas a encontrar el caso contrario, que es uno de los "errores comunes" de la consigna.)

### 2.2 Corregir el tipo de dato de `fecha`

Vimos en la exploración que `fecha` es de tipo `object` (texto), no `datetime`. Además, algunas fechas vienen en formato `dd/mm/yyyy` en vez de `yyyy-mm-dd` (típico cuando distintos sistemas exportan datos). `pd.to_datetime` con `format="mixed"` puede resolver ambos formatos automáticamente.

In [ ]:
# Antes de convertir, veamos un ejemplo de cada formato
print(transacciones["fecha"].sample(10, random_state=3).tolist())


In [ ]:
transacciones["fecha"] = pd.to_datetime(transacciones["fecha"], format="mixed", dayfirst=True)

print(transacciones["fecha"].dtype)
transacciones["fecha"].head()


### 2.3 Manejo de valores nulos

Detectamos nulos en `cantidad`. Como no podemos "inventar" cuántas unidades se vendieron, la decisión de negocio más razonable es **eliminar** esas filas (no imputar una venta que no sabemos si ocurrió). Documentamos cuántas filas se pierden.

In [ ]:
nulos_antes = transacciones["cantidad"].isnull().sum()
transacciones = transacciones.dropna(subset=["cantidad"])
transacciones["cantidad"] = transacciones["cantidad"].astype(int)

print(f"Filas eliminadas por 'cantidad' nula: {nulos_antes}")
print(f"Filas restantes: {len(transacciones)}")


### 2.4 Duplicados

Un error de carga doble es común cuando un sistema reintenta una exportación. Los detectamos y eliminamos.

In [ ]:
duplicados = transacciones.duplicated().sum()
transacciones = transacciones.drop_duplicates()
print(f"Filas duplicadas eliminadas: {duplicados}")


### 2.5 Merge

Unimos ambos datasets usando `id_producto` como llave. Usamos `how="left"` para conservar **todas** las transacciones, incluso si algún producto no está en el catálogo (esto nos permite detectarlo explícitamente, en vez de perder filas silenciosamente con un `inner` join).

In [ ]:
ventas = transacciones.merge(productos, on="id_producto", how="left", indicator=True)

print(ventas["_merge"].value_counts())
ventas.head()


In [ ]:
# Investigamos las transacciones "huérfanas" (producto no encontrado en el catálogo)
huerfanas = ventas[ventas["_merge"] == "left_only"]
print(f"Transacciones con producto no encontrado en el catálogo: {len(huerfanas)}")
huerfanas[["id_transaccion", "id_producto", "cantidad", "fecha"]]


**Decisión de negocio:** estas transacciones probablemente correspondan a productos discontinuados que ya no están en el maestro. Las excluimos del dataset final (no podemos calcular su `total_venta` sin `precio_unitario`), pero las dejamos documentadas como hallazgo — en un caso real, esto ameritaría una alerta al equipo de Compras.

In [ ]:
ventas = ventas[ventas["_merge"] == "both"].drop(columns="_merge").reset_index(drop=True)
print(f"Filas finales tras el merge: {len(ventas)}")


## 3. Normalización

Creamos la columna `total_venta = cantidad * precio_unitario`.

In [ ]:
ventas["total_venta"] = ventas["cantidad"] * ventas["precio_unitario"]
ventas[["id_transaccion", "nombre_producto", "cantidad", "precio_unitario", "total_venta"]].head()


## 4. Validación final

Antes de exportar, chequeamos que el dataset consolidado cumpla condiciones mínimas de calidad.

In [ ]:
assert ventas.isnull().sum().sum() == 0, "Hay nulos en el dataset final"
assert ventas.duplicated().sum() == 0, "Hay duplicados en el dataset final"
assert (ventas["total_venta"] >= 0).all(), "Hay montos negativos, algo está mal"
assert pd.api.types.is_datetime64_any_dtype(ventas["fecha"]), "La fecha no quedó como datetime"

print("Validaciones OK ✅")
print(f"\nResumen final: {len(ventas)} transacciones por un total de ${ventas['total_venta'].sum():,.0f}")
ventas.describe(include="all")


## 5. Exportación a Parquet

Guardamos el resultado consolidado en formato Parquet (más liviano y rápido de leer que CSV/Excel para las siguientes etapas del pipeline).

In [ ]:
ventas.to_parquet(RUTA_SALIDA, index=False)

print(f"Archivo exportado: {RUTA_SALIDA} ({RUTA_SALIDA.stat().st_size / 1024:.1f} KB)")

# Verificación de lectura
verificacion = pd.read_parquet(RUTA_SALIDA)
assert verificacion.equals(ventas)
print("Verificación de lectura OK ✅")


## Cierre — Checklist de la pre-entrega

- [x] Repositorio de GitHub con el notebook, los archivos fuente (`transacciones_electrosur.csv`, `productos_electrosur.xlsx`) y el resultado (`ventas_electrosur_consolidado.parquet`).
- [x] Tipos de datos corregidos (`fecha` como `datetime`).
- [x] Nulos manejados con una decisión justificada (se eliminaron, no se inventó información).
- [x] Merge realizado verificando que las llaves tengan el mismo tipo de dato.
- [x] Columna `total_venta` calculada.
- [x] Exportación a Parquet.
- [ ] **Pendiente de cada alumno:** agregar un `README.md` al repositorio explicando el proceso (uno de los errores comunes es olvidarlo).

➡️ Ahora resolvé el notebook **`Practica_Adquisicion_y_Normalizacion_Libreria.ipynb`** con el dataset de la librería "Página Nueva", que tiene algunos problemas adicionales para que practiques (tipos de datos incompatibles en el ID, fechas inválidas, etc.).